# bayesflow_hpo Quickstart

A minimal end-to-end example using a toy Gaussian simulator and a one-trial optimization run.

In [ ]:
import bayesflow as bf
import bayesflow_hpo as hpo
import numpy as np

In [ ]:
def prior_fn():
    return {"theta": np.random.normal(0.0, 1.0, size=(1,)).astype("float32")}


def likelihood_fn(theta):
    theta_value = float(np.squeeze(theta))
    x = np.random.normal(theta_value, 1.0, size=(12, 1)).astype("float32")
    return {"x": x}


simulator = bf.simulators.make_simulator([prior_fn, likelihood_fn])
adapter = (
    bf.Adapter()
    .as_set(["x"])
    .rename("theta", "inference_variables")
    .concatenate(["x"], into="summary_variables", axis=-1)
)

validation_data = hpo.generate_validation_dataset(
    simulator=simulator,
    param_keys=["theta"],
    data_keys=["x"],
    sims_per_condition=16,
)

In [ ]:
study = hpo.optimize(
    simulator=simulator,
    adapter=adapter,
    param_keys=["theta"],
    data_keys=["x"],
    validation_data=validation_data,
    n_trials=1,
    epochs=1,
    batches_per_epoch=1,
    show_progress_bar=False,
)

print(f"Trials: {len(study.trials)}")
print(f"Best values: {study.best_trials[0].values}")